**Set environment**

In [1]:
source ../run_config_project.sh
show_env

BASE DIRECTORY (FD_BASE):      /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO):      /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK):      /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA):      /hpc/group/igvf/kk319/data
CONTAINER DIR. (FD_SING):      /hpc/group/igvf/kk319/container

You are working with           
PATH OF PROJECT (FD_PRJ):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references
PR

## Preview

In [2]:
ls -1 ${FD_RES}

analysis_variant_motif_richard
analysis_variant_motif_richard_arc251231
analysis_variant_motif_richard_arc260223
predict_variant_alphagenome
predict_variant_kircher2019


In [3]:
ls ${FD_RES}/analysis_variant_motif_richard

background_zero_order.npy
background_zero_order.tsv
batches_dev
batches_pilot_low_dinuc
batches_pilot_low_ori
batches_pilot_top_dinuc
batches_pilot_top_ori
batches_top_dinuc
batches_top_ori
motifdelta_pilot_low_dinuc_jvierstra_v2.1beta
motifdelta_pilot_low_ori_jvierstra_v2.1beta
motifdelta_pilot_top_dinuc_jvierstra_v2.1beta
motifdelta_pilot_top_ori_jvierstra_v2.1beta
motifdelta_top_ori_jaspar2024
motifdelta_top_ori_jvierstra_v2.1beta
motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl
motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
motif_nonredundant_jvierstra_v2.1beta.lods.pkl
motif_nonredundant_jvierstra_v2.1beta.pmap.pkl
motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
motifscan_pilot_low_dinuc_jvierstra_v2.1beta
motifscan_pilot_low_ori_jvierstra_v2.1beta
motifscan_pilot_top_dinuc_jvierstra_v2.1beta
motifscan_pilot_top_ori_jvierstra_v2.1beta
motifscan_top_ori_jaspar2024
motifscan_top_ori_jvierstra_v2.1beta
variant_c

**Set compute resource**

In [4]:
### choose compute profile
CHOOSE_PARTITION="biostat"
#CHOOSE_PARTITION="igvf"
#CHOOSE_PARTITION="igvf_common"
#CHOOSE_PARTITION="biostat_common"

case "$CHOOSE_PARTITION" in
    igvf)
        SLURM_ACCOUNT="majoroslab"
        SLURM_PARTITION="igvf"
        ;;
    igvf_common)
        SLURM_ACCOUNT="majoroslab"
        SLURM_PARTITION="igvf,common"
        ;;
    biostat)
        SLURM_ACCOUNT="biostat"
        SLURM_PARTITION="biostat"
        ;;
    biostat_common)
        SLURM_ACCOUNT="biostat"
        SLURM_PARTITION="biostat,common"
        ;;
    *)
        echo "Unknown CHOOSE_PARTITION: $CHOOSE_PARTITION" >&2
        exit 1
        ;;
esac

### optional: node excludes (ONLY if needed)
EXCLUDE_COMMON="dcc-comp-10,dcc-core-08,dcc-core-53"
EXCLUDE_NODES=""

# Check if "common" is one of the comma-separated partitions
if [[ ",${SLURM_PARTITION}," == *",common,"* ]]; then
  EXCLUDE_NODES="${EXCLUDE_COMMON}"
fi

echo "SLURM_ACCOUNT=${SLURM_ACCOUNT}"
echo "SLURM_PARTITION=${SLURM_PARTITION}"
echo "EXCLUDE_NODES=${EXCLUDE_NODES:-<none>}"

SLURM_ACCOUNT=biostat
SLURM_PARTITION=biostat
EXCLUDE_NODES=<none>


## Execute

### Prepare

In [5]:
FDIRY=${FD_RES}/analysis_variant_motif_richard
ls -1 ${FDIRY}/batches_top_ori
echo
ls -1 ${FDIRY}/batches_top_dinuc

variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top01k.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top10k.ref.fa.gz

variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top01k.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top10k.ref.fa.gz


In [6]:
### helper function
fun_motif_lods() {
  case "$1" in
    jaspar)     echo "motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl" ;;
    jvierstra)  echo "motif_nonredundant_jvierstra_v2.1beta.lods.pkl" ;;
    *) echo "$1" ;;
  esac
}

fun_motif_bind() {
  case "$1" in
    jaspar)     echo "motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl" ;;
    jvierstra)  echo "motif_nonredundant_jvierstra_v2.1beta.tbind.pkl" ;;
    *) echo "$1" ;;
  esac
}

fun_motif_version() {
  case "$1" in
    jaspar)     echo "jaspar2024" ;;
    jvierstra)  echo "jvierstra_v2.1beta" ;;
    *) echo "$1" ;;
  esac
}

### Run script

In [8]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=20G
LST_OPTS=(
    -A "${SLURM_ACCOUNT}"
    -p "${SLURM_PARTITION}"
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

if [[ -n "${EXCLUDE_NODES}" ]]; then
    LST_OPTS+=( --exclude="${EXCLUDE_NODES}" )
fi

### set parameters
FN_PREFIX="variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
LST_MOTIF_LAB=(jaspar jvierstra)
NUM_FLANK_LEFT=35
NUM_BATCH_SIZE=2000

### Loop init
TXT_EXE="motifscan"
TXT_BATCH_SET="top"
LST_SRC=(ori dinuc)
LST_TAGS=(top01k top10k)
LST_JOBS=()

### Loop through I/O
for TXT_MOTIF_LAB in "${LST_MOTIF_LAB[@]}"; do
    ### set motif
    TXT_MOTIF_LODS="$(fun_motif_lods   "${TXT_MOTIF_LAB}")"
    TXT_MOTIF_BIND="$(fun_motif_bind   "${TXT_MOTIF_LAB}")"
    TXT_MOTIF_SET="$(fun_motif_version "${TXT_MOTIF_LAB}")"

    echo ${TXT_MOTIF_LODS}
    echo ${TXT_MOTIF_BIND}
    echo ${TXT_MOTIF_SET}
    echo
    FP_MOTIF="${FD_RES}/analysis_variant_motif_richard/${TXT_MOTIF_LODS}"
    FP_TBIND="${FD_RES}/analysis_variant_motif_richard/${TXT_MOTIF_BIND}"
    
    if [[ ! -f "${FP_MOTIF}" ]]; then
        echo "Missing motif lods: ${FP_MOTIF}" >&2
        continue
    fi

    for TXT_SRC in "${LST_SRC[@]}"; do
        ### Set input/output batch directory based on batch set
        TXT_FOLDER="${TXT_BATCH_SET}_${TXT_SRC}"
        FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_${TXT_FOLDER}
        FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_${TXT_FOLDER}_${TXT_MOTIF_SET}
        FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_${TXT_FOLDER}_${TXT_MOTIF_SET}
        mkdir -p "${FD_MSCAN}"
        mkdir -p "${FD_DELTA}"
    
        for TXT_TAG in "${LST_TAGS[@]}"; do
            ### set job/log
            TXT_JOB="${TXT_EXE}.${TXT_MOTIF_LAB}.${TXT_FOLDER}.${TXT_TAG}"    
            FN_LOG="run.${TXT_JOB}.txt"
            FP_LOG=${FD_LOG}/${FN_LOG}
            
            ### set I/O
            FP_INP=${FD_BATCH}/${FN_PREFIX}.${TXT_TAG}.ref.fa.gz
            FP_OUT=${FD_MSCAN}/${FN_PREFIX}.${TXT_TAG}.npz
            
            ### check file existence
            if [[ ! -f "${FP_INP}" ]]; then
                echo "Missing input: ${FP_INP}"  >&2
                continue
            fi
            
            ### set memory
            if [[ "${TXT_TAG}" == "top01k" ]]; then
                NUM_MEM=8G
            else
                NUM_MEM=40G
            fi
            
            ### execute
            JOBID=$(sbatch \
                "${LST_OPTS[@]}" \
                --mem="${NUM_MEM}" \
                --job-name="${TXT_JOB}" \
                --output="${FP_LOG}" \
                "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}"
            )
            echo "Submitted ${TXT_TAG}: ${JOBID}"
            echo "LOG: \${FD_LOG}/${FN_LOG}"
            LST_JOBS+=("${JOBID}")
        done
        echo
    done
done

motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
jaspar2024

Submitted top01k: 43816878
LOG: ${FD_LOG}/run.motifscan.jaspar.top_ori.top01k.txt
Submitted top10k: 43816879
LOG: ${FD_LOG}/run.motifscan.jaspar.top_ori.top10k.txt

Submitted top01k: 43816880
LOG: ${FD_LOG}/run.motifscan.jaspar.top_dinuc.top01k.txt
Submitted top10k: 43816881
LOG: ${FD_LOG}/run.motifscan.jaspar.top_dinuc.top10k.txt

motif_nonredundant_jvierstra_v2.1beta.lods.pkl
motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
jvierstra_v2.1beta

Submitted top01k: 43816882
LOG: ${FD_LOG}/run.motifscan.jvierstra.top_ori.top01k.txt
Submitted top10k: 43816886
LOG: ${FD_LOG}/run.motifscan.jvierstra.top_ori.top10k.txt

Submitted top01k: 43816887
LOG: ${FD_LOG}/run.motifscan.jvierstra.top_dinuc.top01k.txt
Submitted top10k: 43816888
LOG: ${FD_LOG}/run.motifscan.jvierstra.top_dinuc.top10k.txt



## Review

In [9]:
sacct_summary.sh "${LST_JOBS[@]}"

Detected multiple job IDs (8). Running batch summary.
===== Summary (.ba tasks) =====
JobID             State ExitCode ElapsedRaw   TotalCPU     MaxRSS                       NodeList 
------------ ---------- -------- ---------- ---------- ---------- ------------------------------ 
43816878.ba+  COMPLETED      0:0         37  00:10.037   3172072K                 dcc-biostat-02 
43816879.ba+  COMPLETED      0:0        123  01:18.986  30740524K                 dcc-biostat-02 
43816880.ba+  COMPLETED      0:0         34  00:09.464   3174132K                 dcc-biostat-02 
43816881.ba+  COMPLETED      0:0        122  01:18.751  30741420K                 dcc-biostat-02 
43816882.ba+  COMPLETED      0:0         32  00:05.594   2482444K                 dcc-biostat-02 
43816886.ba+  COMPLETED      0:0         94  00:51.809  23827216K                 dcc-biostat-02 
43816887.ba+  COMPLETED      0:0         33  00:08.246   2480720K                 dcc-biostat-02 
43816888.ba+  COMPLETED      0:0

In [11]:
cat ${FD_LOG}/run.motifscan.jaspar.top_ori.top01k.txt

Hostname:           dcc-biostat-02
Slurm Array Index:  NA
Time Stamp:         02-28-26+18:23:14
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.02 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.00 seconds

Running motif scanning...
Scan complete in 3.17 seconds
Output array size (ref+obs+unobs): 1.454 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_ori_jaspar2024/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top01k.npz
Saved complete in 9.12 seconds


Done!
Run Time: 13 seconds



In [12]:
cat ${FD_LOG}/run.motifscan.jvierstra.top_dinuc.top10k.txt

Hostname:           dcc-biostat-02
Slurm Array Index:  NA
Time Stamp:         02-28-26+18:23:14
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.04 seconds

Loading motif matrices...
Loaded 637 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (637, 28, 4)
Reverse kernels shape: (637, 28, 4)
Set complete in 0.00 seconds

Running motif scanning...
Scan complete in 22.26 seconds
Output array size (ref+obs+unobs): 11.248 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_dinuc_jvierstra_v2.1beta/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top10k.npz
Saved complete in 48.28 seconds


Done!
Run Time: 1 minutes 

In [ ]:
### execute
        JOBID=$(sbatch   \
            "${LST_OPTS[@]}" \
            --mem="${NUM_MEM}" \
            --job-name="${TXT_JOB}" \
            --output="${FP_LOG}"    \
            "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}" \
            | awk '{print $NF}'
        )

In [ ]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=20G
LST_OPTS=(
    -A "${SLURM_ACCOUNT}"
    -p "${SLURM_PARTITION}"
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

if [[ -n "${EXCLUDE_NODES}" ]]; then
    LST_OPTS+=( --exclude="${EXCLUDE_NODES}" )
fi

### set parameters
FN_PREFIX="variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
FN_MOTIF=motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
FN_TBIND=motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/${FN_MOTIF}
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/${FN_TBIND}
TXT_MOTIF_SET=jaspar2024
TXT_MOTIF_LAB=jaspar2024
NUM_FLANK_LEFT=35
NUM_BATCH_SIZE=2000

### Loop init
TXT_EXE="motifscan"
TXT_BATCH_SET="top"
LST_SRC=(ori dinuc)
LST_TAGS=(top01k top10k)
LST_JOBS=()


### Loop through I/O
for TXT_SRC in "${LST_SRC[@]}"; do

    ### Set input/output batch directory based on batch set
    TXT_FOLDER="${TXT_BATCH_SET}_${TXT_SRC}"
    FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_${TXT_FOLDER}
    FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_${TXT_FOLDER}_${TXT_MOTIF_SET}
    FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_${TXT_FOLDER}_${TXT_MOTIF_SET}
    mkdir -p "${FD_MSCAN}"
    mkdir -p "${FD_DELTA}"

    for TXT_TAG in "${LST_TAGS[@]}"; do
        ### set job/log
        TXT_JOB="${TXT_EXE}.${TXT_MOTIF_LAB}.${TXT_FOLDER}.${TXT_TAG}"    
        FN_LOG="run.${TXT_JOB}.txt"
        FP_LOG=${FD_LOG}/${FN_LOG}
        
        ### set I/O
        FP_INP=${FD_BATCH}/${FN_PREFIX}.${TXT_TAG}.ref.fa.gz
        FP_OUT=${FD_MSCAN}/${FN_PREFIX}.${TXT_TAG}.npz
        
        ### check file existence
        if [[ ! -f "${FP_INP}" ]]; then
            echo "Missing input: ${FP_INP}"
            continue
        fi
        
        ### set memory
        if [[ "${TXT_TAG}" == "top01k" ]]; then
            NUM_MEM=8G
        else
            NUM_MEM=40G
        fi
        
        ### execute
        JOBID=$(sbatch   \
            "${LST_OPTS[@]}" \
            --mem="${NUM_MEM}" \
            --job-name="${TXT_JOB}" \
            --output="${FP_LOG}"    \
            "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}" \
            | awk '{print $NF}'
        )
        echo "Submitted ${TXT_TAG}: ${JOBID}"
        echo "LOG: \${FD_LOG}/${FN_LOG}"
        LST_JOBS+=("${JOBID}")
    done
    echo
done

## Execute (JASPAR 2024)

In [ ]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=20G
LST_OPTS=(
    -A "${SLURM_ACCOUNT}"
    -p "${SLURM_PARTITION}"
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

if [[ -n "${EXCLUDE_NODES}" ]]; then
    LST_OPTS+=( --exclude="${EXCLUDE_NODES}" )
fi

### set parameters
FN_PREFIX="variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
FN_MOTIF=motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
FN_TBIND=motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/${FN_MOTIF}
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/${FN_TBIND}
TXT_MOTIF_SET=jaspar2024
TXT_MOTIF_LAB=jaspar2024
NUM_FLANK_LEFT=35
NUM_BATCH_SIZE=2000

### Loop init
TXT_EXE="motifscan"
TXT_BATCH_SET="top"
LST_SRC=(ori dinuc)
LST_TAGS=(top01k top10k)
LST_JOBS=()


### Loop through I/O
for TXT_SRC in "${LST_SRC[@]}"; do

    ### Set input/output batch directory based on batch set
    TXT_FOLDER="${TXT_BATCH_SET}_${TXT_SRC}"
    FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_${TXT_FOLDER}
    FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_${TXT_FOLDER}_${TXT_MOTIF_SET}
    FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_${TXT_FOLDER}_${TXT_MOTIF_SET}
    mkdir -p "${FD_MSCAN}"
    mkdir -p "${FD_DELTA}"

    for TXT_TAG in "${LST_TAGS[@]}"; do
        ### set job/log
        TXT_JOB="${TXT_EXE}.${TXT_MOTIF_LAB}.${TXT_FOLDER}.${TXT_TAG}"    
        FN_LOG="run.${TXT_JOB}.txt"
        FP_LOG=${FD_LOG}/${FN_LOG}
        
        ### set I/O
        FP_INP=${FD_BATCH}/${FN_PREFIX}.${TXT_TAG}.ref.fa.gz
        FP_OUT=${FD_MSCAN}/${FN_PREFIX}.${TXT_TAG}.npz
        
        ### check file existence
        if [[ ! -f "${FP_INP}" ]]; then
            echo "Missing input: ${FP_INP}"
            continue
        fi
        
        ### set memory
        if [[ "${TXT_TAG}" == "top01k" ]]; then
            NUM_MEM=8G
        else
            NUM_MEM=40G
        fi
        
        ### execute
        JOBID=$(sbatch   \
            "${LST_OPTS[@]}" \
            --mem="${NUM_MEM}" \
            --job-name="${TXT_JOB}" \
            --output="${FP_LOG}"    \
            "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}" \
            | awk '{print $NF}'
        )
        echo "Submitted ${TXT_TAG}: ${JOBID}"
        echo "LOG: \${FD_LOG}/${FN_LOG}"
        LST_JOBS+=("${JOBID}")
    done
    echo
done

## Execute: jvierstra

In [ ]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=20G
LST_OPTS=(
    -A "${SLURM_ACCOUNT}"
    -p "${SLURM_PARTITION}"
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

if [[ -n "${EXCLUDE_NODES}" ]]; then
    LST_OPTS+=( --exclude="${EXCLUDE_NODES}" )
fi

### set parameters
FN_PREFIX="variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
FN_MOTIF=motif_nonredundant_jvierstra_v2.1beta.lods.pkl
FN_TBIND=motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/${FN_MOTIF}
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/${FN_TBIND}
TXT_MOTIF_SET=jvierstra_v2.1beta
TXT_MOTIF_LAB=jvierstra
NUM_FLANK_LEFT=35
NUM_BATCH_SIZE=2000

### Loop init
TXT_EXE="motifscan"
TXT_BATCH_SET="top"
LST_SRC=(ori dinuc)
LST_TAGS=(top01k top10k)
LST_JOBS=()

### Loop through I/O
for TXT_SRC in "${LST_SRC[@]}"; do

    ### Set input/output batch directory based on batch set
    TXT_FOLDER="${TXT_BATCH_SET}_${TXT_SRC}"
    FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_${TXT_FOLDER}
    FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_${TXT_FOLDER}_${TXT_MOTIF_SET}
    FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_${TXT_FOLDER}_${TXT_MOTIF_SET}
    mkdir -p "${FD_MSCAN}"
    mkdir -p "${FD_DELTA}"

    for TXT_TAG in "${LST_TAGS[@]}"; do
        ### set job/log
        TXT_JOB="${TXT_EXE}.${TXT_MOTIF_LAB}.${TXT_FOLDER}.${TXT_TAG}"    
        FN_LOG="run.${TXT_JOB}.txt"
        FP_LOG=${FD_LOG}/${FN_LOG}
        
        ### set I/O
        FP_INP=${FD_BATCH}/${FN_PREFIX}.${TXT_TAG}.ref.fa.gz
        FP_OUT=${FD_MSCAN}/${FN_PREFIX}.${TXT_TAG}.npz
        
        ### check file existence
        if [[ ! -f "${FP_INP}" ]]; then
            echo "Missing input: ${FP_INP}"
            continue
        fi

        ### set memory
        if [[ "${TXT_TAG}" == "top01k" ]]; then
            NUM_MEM=8G
        else
            NUM_MEM=40G
        fi
        
        ### execute
        JOBID=$(sbatch   \
            "${LST_OPTS[@]}" \
            --mem="${NUM_MEM}" \
            --job-name="${TXT_JOB}" \
            --output="${FP_LOG}"    \
            "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}" \
            | awk '{print $NF}'
        )
        echo "Submitted ${TXT_TAG}: ${JOBID}"
        echo "LOG: \${FD_LOG}/${FN_LOG}"
        LST_JOBS+=("${JOBID}")
    done
    echo
done

In [ ]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
NUM_MEM=20G
LST_OPTS=(
    -A "${SLURM_ACCOUNT}"
    -p "${SLURM_PARTITION}"
    --cpus-per-task="${NUM_CPU}"
    --mem="${NUM_MEM}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

if [[ -n "${EXCLUDE_NODES}" ]]; then
    LST_OPTS+=( --exclude="${EXCLUDE_NODES}" )
fi

### set parameters
FN_PREFIX="variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
FN_MOTIF=motif_nonredundant_jvierstra_v2.1beta.lods.pkl
FN_TBIND=motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/${FN_MOTIF}
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/${FN_TBIND}
TXT_MOTIF_SET=jvierstra_v2.1beta
TXT_MOTIF_LAB=jvierstra
NUM_FLANK_LEFT=35
NUM_BATCH_SIZE=2000

### Loop init
TXT_EXE="motifscan"
BATCH_SET="top"
LST_SRC=(ori dinuc)
LST_TAGS=(top01k top10k)
LST_JOBS=()

### Loop through I/O
for TXT_SRC in "${LST_SRC[@]}"; do

    ### Set input/output batch directory based on batch set
    TXT_FOLDER="${TXT_BATCH_SET}_${TXT_SRC}"
    FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_${TXT_FOLDER}
    FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_${TXT_FOLDER}_${TXT_MOTIF_SET}
    FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_${TXT_FOLDER}_${TXT_MOTIF_SET}
    mkdir -p "${FD_MSCAN}"
    mkdir -p "${FD_DELTA}"

    for i in $(seq -w 001 020); do
        ### set job/log
        TXT_TAG="${TXT_BATCH_SET}_chunk${i}"
        TXT_JOB="${TXT_EXE}.${TXT_MOTIF_LAB}.${TXT_FOLDER}.${TXT_TAG}"    
        FN_LOG="run.${TXT_JOB}.txt"
        FP_LOG=${FD_LOG}/${FN_LOG}
        
        ### set I/O
        FP_INP=${FD_BATCH}/${FN_PREFIX}.${TXT_TAG}.ref.fa.gz
        FP_OUT=${FD_MSCAN}/${FN_PREFIX}.${TXT_TAG}.npz
        
        ### check file existence
        if [[ ! -f "${FP_INP}" ]]; then
            echo "Missing input: ${FP_INP}"
            continue
        fi
        
        ### execute
        JOBID=$(sbatch   \
            "${LST_OPTS[@]}" \
            --job-name="${TXT_JOB}" \
            --output="${FP_LOG}"    \
            "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}" \
            | awk '{print $NF}'
        )
        echo "Submitted ${TXT_TAG}: ${JOBID}"
        echo "LOG: \${FD_LOG}/${FN_LOG}"
        LST_JOBS+=("${JOBID}")
    done
    echo
done

## Jaspar2024

### Prepare

In [6]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_top
FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_top_jaspar2024
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_top_jaspar2024

FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl

mkdir -p "${FD_MSCAN}"
mkdir -p "${FD_DELTA}"

### Execute

In [7]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=40G
#NUM_MEM=20G

LST_OPTS=(
    -A "${SLURM_ACCOUNT}"
    -p "${SLURM_PARTITION}"
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

### loop init
LST_JOBS=()
LST_TAGS=(top01k top10k)

### Loop through I/O
for TXT_TAG in ${LST_TAGS[@]}; do

    ### set I/O
    TXT_JOB=motifscan.jaspar.${TXT_TAG}
    FP_INP=${FD_BATCH}/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.${TXT_TAG}.ref.fa.gz
    FP_OUT=${FD_MSCAN}/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.${TXT_TAG}.npz
    FP_LOG=${FD_LOG}/run.motifscan.jaspar.batch.${TXT_TAG}.txt
    #FP_LOG=${FD_LOG}/run.motifscan.jaspar.batch.${TXT_TAG}.%j.txt
    NUM_FLANK_LEFT=35
    NUM_BATCH_SIZE=2000
    
    ### set memory
    if [[ "${TXT_TAG}" == "top01k" ]]; then
        NUM_MEM=8G
    else
        NUM_MEM=40G
    fi
    
    ### execute
    JOBID=$(sbatch   \
        "${LST_OPTS[@]}" \
        --mem="${NUM_MEM}" \
        --job-name="${TXT_JOB}" \
        --output="${FP_LOG}"    \
        "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}"
    )
    echo "Submitted ${TXT_TAG}: ${JOBID}"
    LST_JOBS+=("${JOBID}")
done

Submitted top01k: 43674170
Submitted top10k: 43674171


## Review

In [8]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
43674170.ba+                          batch  COMPLETED   00:00:19  00:08.124   3170832K                 dcc-biostat-20 

===== ElapsedRaw =====
ElapsedRaw = 19 sec (0.32 min)

===== MaxRSS =====
MaxRSS = 3.02 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
43674171.ba+                          batch  COMPLETED   00:01:06  00:46.927  30744764K                 dcc-biostat-20 

===== ElapsedRaw =====
ElapsedRaw = 66 sec (1.10 min)

===== MaxRSS =====
MaxRSS = 29.32 GiB



In [9]:
#cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[0]}.txt

Hostname:           dcc-biostat-20
Slurm Array Index:  NA
Time Stamp:         02-23-26+18:29:14
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 3.53 seconds
Output array size (ref+obs+unobs): 1.454 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top01k.npz
Saved complete in 7.38 seconds


Done!
Run Time: 12 seconds



In [10]:
#cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt
cat ${FD_LOG}/run.motifscan.jaspar.batch.${LST_TAGS[1]}.txt

Hostname:           dcc-biostat-20
Slurm Array Index:  NA
Time Stamp:         02-23-26+18:29:13
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.03 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.00 seconds

Running motif scanning...
Scan complete in 22.21 seconds
Output array size (ref+obs+unobs): 14.539 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top10k.npz
Saved complete in 37.07 seconds


Done!
Run Time: 1 minutes and 0

## Non-redundant motifs

### Prepare

In [11]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_top
FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_top_jvierstra_v2.1beta
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_top_jvierstra_v2.1beta

FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.tbind.pkl

mkdir -p "${FD_MSCAN}"
mkdir -p "${FD_DELTA}"

### Execute

In [12]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=40G
#NUM_MEM=20G

LST_OPTS=(
    -A "${SLURM_ACCOUNT}"
    -p "${SLURM_PARTITION}"
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

### loop init
LST_JOBS=()
LST_TAGS=(top01k top10k)

### Loop through I/O
for TXT_TAG in ${LST_TAGS[@]}; do

    ### set I/O
    TXT_JOB=motifscan.jvierstra.${TXT_TAG}
    FP_INP=${FD_BATCH}/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.${TXT_TAG}.ref.fa.gz
    FP_OUT=${FD_MSCAN}/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.${TXT_TAG}.npz
    FP_LOG=${FD_LOG}/run.motifscan.jvierstra.batch.${TXT_TAG}.txt
    #FP_LOG=${FD_LOG}/run.motifscan.jvierstra.batch.${TXT_TAG}.%j.txt
    NUM_FLANK_LEFT=35
    NUM_BATCH_SIZE=2000
    
    ### set memory
    if [[ "${TXT_TAG}" == "top01k" ]]; then
        NUM_MEM=8G
    else
        NUM_MEM=40G
    fi
    
    ### execute
    JOBID=$(sbatch   \
        "${LST_OPTS[@]}" \
        --mem="${NUM_MEM}" \
        --job-name="${TXT_JOB}" \
        --output="${FP_LOG}"    \
        "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}"
    )
    echo "Submitted ${TXT_TAG}: ${JOBID}"
    LST_JOBS+=("${JOBID}")
done

Submitted top01k: 43674202
Submitted top10k: 43674203


## Review

In [13]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
43674202.ba+                          batch  COMPLETED   00:00:15  00:05.945   2477420K                 dcc-biostat-20 

===== ElapsedRaw =====
ElapsedRaw = 15 sec (0.25 min)

===== MaxRSS =====
MaxRSS = 2.36 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- ---------- ---------- ---------- ------------------------------ 
43674203.ba+                          batch  COMPLETED   00:01:00  00:35.330  23820160K                 dcc-biostat-20 

===== ElapsedRaw =====
ElapsedRaw = 60 sec (1.00 min)

===== MaxRSS =====
MaxRSS = 22.72 GiB



In [14]:
#cat ${FD_LOG}/run.motifscan.jvierstra.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt
cat ${FD_LOG}/run.motifscan.jvierstra.batch.${LST_TAGS[0]}.txt

Hostname:           dcc-biostat-20
Slurm Array Index:  NA
Time Stamp:         02-23-26+18:31:01
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 1000 sequences
Load and check complete in 0.01 seconds

Loading motif matrices...
Loaded 637 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (637, 28, 4)
Reverse kernels shape: (637, 28, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 2.12 seconds
Output array size (ref+obs+unobs): 1.125 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jvierstra_v2.1beta/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top01k.npz
Saved complete in 3.99 seconds


Done!
Run Time: 7 seconds



In [15]:
#cat ${FD_LOG}/run.motifscan.jvierstra.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt
cat ${FD_LOG}/run.motifscan.jvierstra.batch.${LST_TAGS[1]}.txt

Hostname:           dcc-biostat-20
Slurm Array Index:  NA
Time Stamp:         02-23-26+18:31:01
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 10000 sequences
Load and check complete in 0.03 seconds

Loading motif matrices...
Loaded 637 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
Load and check complete in 0.00 seconds

Setting motif kernel...
Forward kernels shape: (637, 28, 4)
Reverse kernels shape: (637, 28, 4)
Set complete in 0.00 seconds

Running motif scanning...
Scan complete in 14.70 seconds
Output array size (ref+obs+unobs): 11.248 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jvierstra_v2.1beta/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top10k.npz
Saved complete in 36.72 seconds


Done!
Run Time: 52 seconds

